# W6D2 — Modern LLMs & RAG: Ground the model in your own documents — Lab

**Week 6 · Day 2 · LLMs** · Lab

> **The big idea.** An LLM only knows what was in its pretraining data. For
> anything newer, private, or domain-specific, the fix is **not** to retrain the
> model — it is to **retrieve the relevant text and paste it into the prompt**
> before asking the question. That pattern — retrieve, then generate — is called
> **RAG**, and it is the single most-used LLM deployment pattern in production
> today.

Yesterday you measured a small LM. Today you combine two models (a tiny embedder
and a tiny chat LLM) into a pipeline that answers questions about a corpus the
LLM has never seen. By the end you will have **built RAG from scratch** without
LangChain, LlamaIndex, or any black-box helper — four short functions and a
numpy array for the vector store.

<div dir="rtl" align="right">

> **الفكرة الكبرى.** النموذج اللغويّ لا يعرف إلا ما كان في بيانات تدريبه. لكلّ
> ما هو أحدث أو خاصّ أو متخصّص، الحلّ **ليس** إعادة التدريب — بل **استرجاع النصّ
> ذي الصلة ولصقه في المُحثّ** قبل السؤال. هذا النمط — استرجاع ثم توليد — يُسمّى
> **RAG**، وهو النمط الأكثر استخدامًا في نشر LLMs في الإنتاج اليوم.

بالأمس قست نموذجًا صغيرًا. اليوم تجمع نموذجين (مُضمِّن صغير ونموذج محادثة صغير) في
خطّ أنابيب يُجيب عن أسئلة حول مجموعة نصوص لم يرها النموذج قطّ. في النهاية ستكون قد
**بنيت RAG من الصفر** بدون LangChain أو LlamaIndex — أربع دوال قصيرة ومصفوفة numpy.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

Concretely, you can:

- Turn a document into **chunks**, turn chunks into **embedding vectors**, store
  them in memory, and retrieve the top-k most relevant chunks for a query by
  **cosine similarity**.
- Compose a **grounded prompt** that places the retrieved chunks in front of the
  LLM so its answer sticks to the evidence.
- Explain and demonstrate when a plain LLM **hallucinates** vs when a RAG
  pipeline grounds it.
- Compare **with-RAG vs no-RAG** on the same question and read the difference in
  the output.
- Spot the two failure modes of RAG: (a) the question is **not** in the corpus
  (model should say "I don't know") and (b) **k** is set too high (irrelevant
  chunks dilute the answer).

<div dir="rtl" align="right">

## الأهداف التعلّمية

- أن تُجزّئ وثيقة إلى **مقاطع**، تُحوّلها إلى **متجهات تضمين**، تحفظها في الذاكرة،
  وتسترجع أعلى k مقطع لاستعلام ما بـ**تشابه جيب التمام**.
- أن تُركّب **مُحثًّا مُرسَّخًا** يضع المقاطع المسترجَعة أمام النموذج ليلتزم إجابته بها.
- أن تشرح وتُظهر متى يُهلوس نموذج عاديّ ومتى يُرسّخ RAG الإجابة.
- أن تقارن **مع-RAG مقابل بدون-RAG** على نفس السؤال.
- أن ترصد وضعَي فشل RAG: (أ) السؤال **ليس** في المجموعة (يجب على النموذج أن يقول
  "لا أعلم") و (ب) **k** عالٍ جدًّا (مقاطع غير ذات صلة تُشوّش الإجابة).

</div>


## About the model and data

**Generator (chat LLM):** `Qwen/Qwen2.5-0.5B-Instruct` — 494M parameters,
instruction-tuned by Alibaba, decent at both English and Arabic, fits in ~1 GB
RAM, runs on Colab CPU in a few seconds per answer.

**Embedder:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` —
118M parameters, trained by UKP + sbert.net on paraphrase pairs across 50+
languages (including Arabic). Maps any sentence to a shared 384-dimensional
space so cosine similarity works across languages.

**Data:** **10 short English snippets** from a fictional "Olo Academy
Handbook" — course policies, instructor names, deadlines, a few quirky facts.
All inline in the notebook (no download, no API).

Fictional on purpose: because none of these facts could have been in Qwen's
pretraining data, you will see RAG's effect **cleanly** — the bare LLM
hallucinates confidently, the RAG pipeline grounds the answer in the correct
snippet. In production you'd swap this list for your own PDFs, wiki pages,
Confluence export, support tickets, etc. — the pipeline is identical.

<div dir="rtl" align="right">

## عن النموذج والبيانات

**المُولِّد (نموذج محادثة):** `Qwen/Qwen2.5-0.5B-Instruct` — ٤٩٤M معامل، مُدرَّب
على التعليمات، جيّد بالعربيّة والإنجليزيّة، يسع في ~١ GB، يعمل على Colab CPU.

**المُضمِّن:** `paraphrase-multilingual-MiniLM-L12-v2` — ١١٨M معامل، مُدرَّب على
أزواج إعادة صياغة بأكثر من ٥٠ لغة (ومنها العربيّة). يُحوّل أي جملة إلى فضاء
مشترك بـ٣٨٤ بُعدًا.

**البيانات:** **١٠ مقاطع إنجليزيّة قصيرة** من "دليل أكاديمية أولو" الخياليّ —
سياسات ومواعيد وحقائق. كلها ضمن الخلايا (لا تنزيل ولا API).

خياليّة عمدًا: لأن هذه الحقائق لم تكن في بيانات تدريب Qwen، سترى أثر RAG
**بوضوح** — النموذج العاري يُهلوس بثقة، وخطّ RAG يُرسّخ الإجابة في المقطع
الصحيح. في الإنتاج ستُبدِّل القائمة بـPDFs أو صفحات ويكي أو تذاكر دعم —
الخطّ نفسه.

</div>


## Setup

Install (on Colab) and import. Then load the two models. This takes ~30 seconds
the first time (downloading ~700 MB total) and is instant after that.

<div dir="rtl" align="right">

## الإعداد

ثبّت (على Colab) واستورد. ثم حمّل النموذجين. أول مرّة ~٣٠ ثانية (~٧٠٠ MB)، وفوريّ
بعدها.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.40", "sentence-transformers>=3.0",
                           "accelerate"])

import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}  device={DEVICE}")

In [ ]:
# === Load the embedder and the chat LLM =============================================
EMBED_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"

embedder  = SentenceTransformer(EMBED_NAME, device=DEVICE)
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm       = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32).to(DEVICE)
llm.eval()

print(f"Embedder: {EMBED_NAME}  (dim={embedder.get_sentence_embedding_dimension()})")
print(f"LLM:      {LLM_NAME}  ({sum(p.numel() for p in llm.parameters())/1e6:.0f}M params)")

## Section 1 — Warm-up: two primitives  *(~10 min)*

Nothing to implement. You'll use two helpers everywhere in Section 2 — a quick
look so there are no surprises:

1. **`embed(texts)`** — turn one or many strings into unit-norm vectors.
2. **`chat(prompt)`** — one-shot answer from the instruct-tuned LLM.

<div dir="rtl" align="right">

## القسم ١ — إحماء: أداتان  *(≈١٠ دقائق)*

لا شيء تنفّذه. ستستخدم في القسم ٢ هاتين الدالّتين — نلقي نظرة سريعة حتى لا توجد
مفاجآت:

١. **`embed(texts)`** — تحوّل نصوصًا إلى متجهات مُوحَّدة الطول.
٢. **`chat(prompt)`** — إجابة من النموذج المُدرَّب على التعليمات.

</div>


In [ ]:
# === Primitive 1: embed one or many strings into unit-norm vectors ==================
def embed(texts):
    """Return a numpy array of shape (N, D) with L2-normalized rows."""
    if isinstance(texts, str):
        texts = [texts]
    return embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

# Demo: similarity between related and unrelated sentences
pairs = [
    ("A cat sits on the mat.",  "A kitten rests on the rug."),   # same meaning
    ("A cat sits on the mat.",  "Transformers process sequences."),  # unrelated
]
for a, b in pairs:
    va, vb = embed([a, b])
    sim = float(np.dot(va, vb))   # cosine, since both are unit-norm
    print(f"sim={sim:+.2f}   '{a}' vs '{b}'")

In [ ]:
# === Primitive 2: one-shot chat with the instruct LLM ===============================
@torch.no_grad()
def chat(user_message, system_message="You are a helpful, concise assistant.",
         max_new_tokens=120):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.strip()

print(chat("What is the capital of Saudi Arabia?"))

Two helpers, two primitives. In Section 2 you combine them: use `embed` to
retrieve the right chunks from a corpus, then feed those chunks to `chat` as
context.

<div dir="rtl" align="right">

دالّتان، أساسيّتان. في القسم ٢ تجمعهما: `embed` لاسترجاع المقاطع الصحيحة من
المجموعة، ثم تمرّرها إلى `chat` كسياق.

</div>


## Section 2 — Build the RAG pipeline  *(~55 min)*

Five tasks, each one new piece of the pipeline. By Task 5 you have an end-to-end
system that answers questions about the Olo Academy Handbook.

1. **Chunk the corpus** — split documents into retrievable units.
2. **Embed the chunks** — build an in-memory vector store (just a numpy matrix).
3. **Retrieve top-k** — cosine similarity, no vector database needed.
4. **Compose a grounded prompt** — plug chunks into a template, ask the LLM.
5. **Compare with-RAG vs no-RAG** — two questions, two prompts each, read the
   difference.

<div dir="rtl" align="right">

## القسم ٢ — ابنِ خطّ RAG  *(≈٥٥ دقيقة)*

خمس مهام، كلّ واحدة جزء جديد من الخطّ. في المهمّة ٥ سيكون لديك نظام كامل يُجيب عن
أسئلة حول دليل أكاديمية أولو.

١. **جزّئ المجموعة** — قسّم الوثائق إلى وحدات قابلة للاسترجاع.
٢. **ضمّن المقاطع** — ابنِ مخزن متّجهات في الذاكرة (مصفوفة numpy فقط).
٣. **استرجع أعلى k** — تشابه جيب التمام، لا حاجة لقاعدة متّجهات.
٤. **ركّب مُحثًّا مُرسَّخًا** — ضع المقاطع في قالب واسأل النموذج.
٥. **قارن مع-RAG وبدون-RAG** — سؤالان، مُحثّان لكل سؤال، اقرأ الفرق.

</div>


In [ ]:
# === The corpus: 10 fictional Olo Academy Handbook snippets =========================
DOCUMENTS = [
    "The AI Engineering Bootcamp (AIEP) at Olo Academy is an 8-week intensive "
    "program covering foundations, data engineering, neural networks, CNNs, NLP, "
    "LLMs, retrieval, and a final capstone project.",

    "Students at Olo Academy receive a certificate of completion only if they "
    "submit all 5 weekly projects and the final capstone with a passing grade.",

    "The lead instructor for the AIEP program is Ghaida Alshehri. She holds "
    "office hours every Tuesday and Thursday from 2 PM to 4 PM Riyadh time.",

    "All lab notebooks are run on Google Colab or locally in VSCode using the "
    "shared AIEP conda environment. GPU access is provided through Colab's free T4 tier.",

    "Olo Academy's cohort of October 2026 is capped at 40 students. Applications "
    "close one week before the start date and require a short technical interview.",

    "The AIEP capstone project runs during week 8 and is presented live on the "
    "final Thursday to a panel of three industry mentors.",

    "Olo Academy was founded in 2024 and is headquartered in Riyadh, Saudi Arabia. "
    "The academy focuses exclusively on applied AI engineering education.",

    "Students are expected to complete approximately 20 hours of self-study per "
    "week, in addition to the 15 hours of live instruction and labs.",

    "The Olo Academy Discord server is the primary channel for peer support. "
    "Instructors monitor the #help channel from 9 AM to 9 PM Riyadh time daily.",

    "Attendance below 70% of live sessions results in automatic removal from the "
    "program, regardless of project grades. There are no exceptions.",
]
print(f"Loaded {len(DOCUMENTS)} documents, average length: "
      f"{sum(len(d) for d in DOCUMENTS)/len(DOCUMENTS):.0f} chars")

### Task 1 — Chunk the corpus

The documents are already short, but in real deployments you receive long PDFs,
wiki pages, or transcripts. Standard practice is to split them into fixed-size
**chunks** (usually a few hundred characters) so each retrieved piece fits in
the LLM prompt comfortably.

Here, since each document is already a self-contained paragraph, write a trivial
chunker that returns one chunk per document. Return a list of
`(chunk_text, doc_id)` pairs so you can trace any answer back to its source.

<div dir="rtl" align="right">

### المهمّة ١ — جزّئ المجموعة

الوثائق قصيرة أصلًا، لكن في النشر الفعليّ تستقبل PDFs أو صفحات ويكي طويلة.
الممارسة المعتادة تجزئتها إلى **مقاطع** بحجم ثابت.

هنا، بما أن كل وثيقة فقرة قائمة بذاتها، اكتب مُجزّئًا بسيطًا يُرجع مقطعًا واحدًا لكل
وثيقة. أرجع قائمة أزواج `(نص المقطع، معرّف الوثيقة)` لتتبّع مصدر كل إجابة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The function takes a list of documents and returns a list of (text, doc_id) tuples.
# 2) Use enumerate() to get the doc_id (an int) alongside each document.
# 3) For now, one chunk per document is fine. You could split long docs by paragraph later.
# 4) Expected output: a list of length 10 with (chunk_text, 0..9) pairs.
#
# ١) الدالّة تأخذ قائمة وثائق وتُرجع قائمة أزواج (نص، معرّف الوثيقة).
# ٢) استخدم enumerate() للحصول على المعرّف.
# ٣) الآن يكفي مقطع واحد لكل وثيقة.
# ٤) المتوقّع: قائمة بطول ١٠ مع أزواج (نص_المقطع، ٠..٩).
# ────────────────────────────────────────────────────────────────────

    # TODO: Implement chunk_corpus to return [(text, doc_id), ...] pairs.
    # مهمة: نفّذ chunk_corpus لتُرجع أزواج [(نص، معرّف_الوثيقة)، ...].

### Task 2 — Embed the chunks

Call `embed()` on just the text part of each chunk and store the result as a
single `(N, D)` numpy matrix. This is your "vector store". In production you'd
use ChromaDB, FAISS, or pgvector — but for teaching, a numpy matrix is enough
and much easier to reason about.

<div dir="rtl" align="right">

### المهمّة ٢ — ضمّن المقاطع

نادِ `embed()` على نصّ كل مقطع واحفظ النتيجة كمصفوفة numpy واحدة بشكل `(N, D)`.
هذا هو "مخزن المتّجهات" لديك.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Extract just the text part: [c[0] for c in CHUNKS] or (c[0] for c in CHUNKS).
# 2) Pass that list to embed(). It returns an (N, D) numpy array already.
# 3) Expected shape: (10, 384) because MiniLM-L12 outputs 384-dim vectors.
#
# ١) استخرج النصّ فقط: [c[0] for c in CHUNKS].
# ٢) مرّر القائمة لـembed() — تُرجع مصفوفة (N, D) جاهزة.
# ٣) الشكل المتوقّع: (10, 384) لأن MiniLM-L12 يُنتج متجهات ٣٨٤ بُعدًا.
# ────────────────────────────────────────────────────────────────────

# TODO: Embed every chunk's text into a single (N, D) numpy matrix.
# مهمة: ضمّن نصّ كل مقطع في مصفوفة numpy واحدة بشكل (N, D).

### Task 3 — Retrieve the top-k chunks for a query

Given a query string, embed it and find the `k` chunks with the highest cosine
similarity. Since both the query and the chunk vectors are already
unit-normalised, cosine similarity reduces to a plain **dot product**:

$$\text{cos}(q, c) = q \cdot c$$

Return the top-k results as a list of `(score, chunk_text, doc_id)` tuples
sorted by score descending.

<div dir="rtl" align="right">

### المهمّة ٣ — استرجع أعلى k مقاطع لاستعلام ما

أعطَى استعلامًا، ضمّنه وجد المقاطع الـk الأعلى تشابهًا جيبيًّا. بما أن كلًا من
الاستعلام والمقاطع مُوحَّد الطول، فتشابه جيب التمام يصبح **ضرب نقطيّ** مباشرة.

أرجع أعلى k كقائمة أزواج `(الدرجة، نصّ المقطع، معرّف الوثيقة)` مرتَّبة تنازليًّا.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) q = embed(query)[0]  # drop the leading batch dim to get a (D,) vector.
# 2) scores = CHUNK_VECTORS @ q  # (N,) vector of similarities.
# 3) top_k_idx = np.argsort(scores)[::-1][:k]  # k highest indices.
# 4) For each index i, assemble (float(scores[i]), CHUNKS[i][0], CHUNKS[i][1]).
#
# ١) q = embed(query)[0]  # أزِل بُعد الدفعة لتحصل على متجه (D,).
# ٢) scores = CHUNK_VECTORS @ q  # متجه تشابه (N,).
# ٣) top_k_idx = np.argsort(scores)[::-1][:k]  # k أعلى فهارس.
# ٤) لكل فهرس i، كوّن (float(scores[i])، CHUNKS[i][0]، CHUNKS[i][1]).
# ────────────────────────────────────────────────────────────────────

    # TODO: Implement retrieve(query, k) returning the top-k (score, text, doc_id) tuples.
    # مهمة: نفّذ retrieve(query, k) لتُرجع أعلى k أزواج (الدرجة، النصّ، معرّف الوثيقة).

### Task 4 — Compose a grounded prompt

Now the glue: a function `rag_answer(question, k)` that retrieves the top-k
chunks, puts them in a prompt template, and asks the LLM. The template should:

- Clearly label the retrieved context.
- Instruct the model to answer *only* from the context, and to say "I don't
  know" if the answer is not there.
- Place the question at the end so the LLM focuses on it.

<div dir="rtl" align="right">

### المهمّة ٤ — ركّب مُحثًّا مُرسَّخًا

الآن اللاصق: دالّة `rag_answer(question, k)` تسترجع أعلى k مقاطع، تضعها في قالب
مُحثّ، وتسأل النموذج. القالب ينبغي أن:

- يُسمّي السياق المسترجَع بوضوح.
- يُوجِّه النموذج ليُجيب *فقط* من السياق، ويقول "لا أعلم" إذا لم تكن الإجابة فيه.
- يضع السؤال في النهاية.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Call retrieve(question, k) to get the top-k chunks.
# 2) Build the context string by numbering each chunk, e.g. "[1] <text>\n[2] <text>".
# 3) Compose: "Context:\n<context>\n\nAnswer the question using ONLY the context above."
# 4) If the context does not contain the answer, say 'I don't know.'. Then: Question: <q>".
# 5) Return chat(the composed prompt). You already have the chat() helper from Section 1.
#
# ١) نادِ retrieve(question, k) للحصول على أعلى k مقاطع.
# ٢) ابنِ سياقًا بترقيم كل مقطع.
# ٣) ركّب: "Context:\n...\n\nAnswer the question using ONLY the context above."
# ٤) إذا لم تكن الإجابة في السياق، قُل 'I don't know.'. ثم: "Question: <q>".
# ٥) أرجع chat(المُحثّ المُركّب). دالّة chat() لديك من القسم ١.
# ────────────────────────────────────────────────────────────────────

    # TODO: Implement rag_answer(question, k) that retrieves k chunks and calls chat().
    # مهمة: نفّذ rag_answer(question, k) التي تسترجع k مقاطع وتنادي chat().

### Task 5 — Compare with-RAG vs no-RAG

Two questions. Each one asked **twice**:
- Once directly to `chat()` with no context (bare LLM).
- Once through `rag_answer()` with k=3.

Print both answers side-by-side. The first question has an answer in the
corpus; the second one asks for a very specific detail (office hours).
Expect the bare LLM to hallucinate confidently, and the RAG answer to be
specific and correct.

<div dir="rtl" align="right">

### المهمّة ٥ — قارن مع-RAG وبدون-RAG

سؤالان. كلّ واحد يُسأل **مرّتين**:
- مرّة مباشرة لـ`chat()` بلا سياق.
- مرّة عبر `rag_answer()` بـk=3.

اطبع الإجابتين جنبًا إلى جنب. توقّع هلوسة من النموذج العاري، وإجابة محدّدة
وصحيحة من RAG.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) For each question, call chat(question) to get the no-RAG answer.
# 2) Then call rag_answer(question, k=3) to get the RAG answer and sources.
# 3) Print both so they are easy to compare visually.
# 4) Expect: no-RAG makes up names/times; RAG uses the chunks and is correct.
#
# ١) لكل سؤال، نادِ chat(question) للإجابة بدون-RAG.
# ٢) ثم rag_answer(question, k=3) للإجابة بـRAG والمصادر.
# ٣) اطبع الإجابتين لتسهيل المقارنة.
# ٤) توقّع: بدون-RAG يختلق أسماءً/أوقاتًا؛ مع-RAG يستخدم المقاطع وصحيح.
# ────────────────────────────────────────────────────────────────────

# TODO: For two questions, show the no-RAG vs with-RAG answers side by side.
# مهمة: لسؤالَين، أظهر الإجابة بدون-RAG مقابل مع-RAG جنبًا إلى جنب.

### Takeaway

RAG does not make the LLM smarter — it **changes what it has in front of it**.
The LLM is doing the same next-token prediction it did yesterday. What's new is
the **retriever** that fills the prompt with evidence, and the **grounding
instruction** that tells the LLM to stay inside it.

This is exactly what every production LLM app with "knowledge of your docs"
does under the hood, from Perplexity to enterprise search to customer support
bots. You just built the whole thing in 4 functions.

<div dir="rtl" align="right">

### الخلاصة

RAG لا يجعل النموذج أذكى — بل **يُغيّر ما أمامه**. النموذج يُجري نفس التنبّؤ
التالي بالرمز الذي قام به بالأمس. الجديد هو **المسترجِع** الذي يملأ المُحثّ
بأدلّة، و**تعليمة الترسيخ** التي تطلب منه البقاء داخلها.

هذا بالضبط ما تفعله كل تطبيقات LLM الإنتاجيّة "التي تعرف وثائقك" — من Perplexity
إلى بحث المؤسّسات إلى بوتات دعم العملاء. وقد بنيتَه كلّه في أربع دوال.

</div>


## Section 3 — Stress-test the pipeline  *(~25 min)*

Section 2 showed RAG when the answer *is* in the corpus and `k` is reasonable.
Real pipelines fail quietly in two ways:

- The answer **is not in the corpus**. A well-prompted LLM should say "I don't
  know" — but will it?
- **k is too large** — irrelevant chunks drown out the relevant ones and the
  LLM gets confused.

Two tasks + a short write-up. This section is where you build intuition for
when RAG works and when it does not.

<div dir="rtl" align="right">

## القسم ٣ — اختبر الخطّ تحت الضغط  *(≈٢٥ دقيقة)*

القسم ٢ أظهر RAG عندما تكون الإجابة *في* المجموعة وk معقول. الخطوط الفعليّة
تفشل بصمت بطريقتين:

- الإجابة **ليست** في المجموعة. النموذج المُوجَّه جيّدًا ينبغي أن يقول "لا أعلم" —
  هل سيفعل؟
- **k كبير جدًّا** — المقاطع غير ذات الصلة تُغرق المقاطع ذات الصلة ويرتبك النموذج.

مهمّتان + كتابة قصيرة. هنا تبني حدسك لمتى يعمل RAG ومتى لا.

</div>


### Task 1 — Ask an out-of-corpus question

Pick a question whose answer is **definitely not** in `DOCUMENTS` (e.g., ask
about tuition fees, which the handbook never mentions). Call `rag_answer()`.

Does the LLM honour the "say 'I don't know'" instruction? Or does it still try
to answer from the irrelevant retrieved chunks? Note which, because this is the
single most important failure mode of RAG in production.

<div dir="rtl" align="right">

### المهمّة ١ — اسأل سؤالًا خارج المجموعة

اختر سؤالًا إجابته **ليست** في `DOCUMENTS` (مثلًا عن الرسوم الدراسيّة، لم يذكرها
الدليل). نادِ `rag_answer()`.

هل يحترم النموذج تعليمة "قل لا أعلم"؟ أم يحاول الإجابة من المقاطع غير ذات
الصلة؟ لاحظ، فهذا أهمّ وضع فشل لـRAG في الإنتاج.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Pick a question definitely NOT answered by the 10 handbook snippets.
# 2) Call rag_answer(q, k=3) and inspect what came back AND which chunks were used.
# 3) Compare: did the LLM say "I don't know" or did it invent a tuition fee?
# 4) The retrieved chunks will likely be semantically-closest-but-wrong.
#
# ١) اختر سؤالًا الإجابة ليست في مقاطع الدليل العشرة.
# ٢) نادِ rag_answer(q, k=3) وافحص ما جاء والمقاطع المستخدمة.
# ٣) قارن: هل قال "لا أعلم" أم اختلق رسومًا دراسيّة؟
# ٤) المقاطع المسترجَعة ستكون الأقرب دلاليًّا لكنّها خاطئة.
# ────────────────────────────────────────────────────────────────────

# TODO: Ask an out-of-corpus question through rag_answer() and inspect behaviour.
# مهمة: اسأل سؤالًا خارج المجموعة عبر rag_answer() وافحص السلوك.

### Task 2 — Vary `k` and watch the answer change

Pick one question whose answer *is* in the corpus. Call `rag_answer()` with
`k = 1, 3, 6, 10` and print the answers.

- At **k=1** you give the LLM one chunk — if retrieval was wrong, you lose.
- At **k=3** (the default) you usually have the right chunk plus some noise.
- At **k=10** you hand the LLM *everything* and ask it to find the answer
  itself. Watch the quality degrade: more context ≠ better answer.

<div dir="rtl" align="right">

### المهمّة ٢ — نوّع `k` وراقب تغيّر الإجابة

اختر سؤالًا إجابته *في* المجموعة. نادِ `rag_answer()` بـ`k = 1, 3, 6, 10` واطبع
الإجابات.

- عند **k=1**: مقطع واحد — إذا أخطأ الاسترجاع، تخسر.
- عند **k=3** (الافتراضيّ): غالبًا المقطع الصحيح + ضوضاء.
- عند **k=10**: تسلّم النموذج *كلّ شيء* وتطلب منه إيجاد الإجابة. راقب تدهور
  الجودة: مزيد من السياق ≠ إجابة أفضل.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Loop k over [1, 3, 6, 10] and call rag_answer(question, k=k) each time.
# 2) Print k and the answer. Ignore sources this time — focus on the output text.
# 3) Expect quality to peak around k=3 and get noisier at k=10.
#
# ١) كرّر k في [1, 3, 6, 10] ونادِ rag_answer(question, k=k) في كل مرّة.
# ٢) اطبع k والإجابة. تجاهل المصادر — ركّز على النصّ.
# ٣) توقّع ذروة الجودة عند k=3 وضجيجًا عند k=10.
# ────────────────────────────────────────────────────────────────────

# TODO: Ask the same question with k=1, 3, 6, 10. Observe how the answer changes.
# مهمة: اسأل نفس السؤال بـk=1, 3, 6, 10. لاحظ تغيّر الإجابة.

### Write-up

In **one English sentence**, write down the rule-of-thumb you now have for
choosing `k` in a RAG system. (Hint: it depends on how clean your retrieval
is and how long each chunk is.)

> *Example answer: "Keep k just large enough that the right chunk is almost
> always in the set — raising it further costs tokens and dilutes the LLM's
> focus without adding information."*

<div dir="rtl" align="right">

### كتابة

في **جملة إنجليزيّة واحدة**، اكتب القاعدة التجريبيّة لاختيار `k` في نظام RAG.
(تلميح: تعتمد على نظافة الاسترجاع وطول كل مقطع.)

</div>


## Wrap-up

Four things you can now do that you couldn't an hour ago:

1. **Build a RAG pipeline from scratch** in four functions — no LangChain, no
   framework, no vector database.
2. **Explain and demonstrate** the two failure modes of RAG (out-of-corpus
   question, k too large) with your own working examples.
3. **Choose k** for a RAG system based on the retrieval quality and the chunk
   size, instead of defaulting to whatever a tutorial said.
4. **Compare** a bare LLM answer against a grounded one and read which one is
   trustworthy for a given query.

Tomorrow in **D3** we take the next step: instead of giving the LLM retrieved
text to read, we give it **tools to use** — a calculator, a search engine, a
file reader — and let it plan a sequence of actions. That is an agent.

<div dir="rtl" align="right">

## الخلاصة

أربعة أمور تستطيع الآن:

١. **تبني خطّ RAG من الصفر** في أربع دوال — بلا LangChain ولا إطار ولا قاعدة متّجهات.
٢. **تشرح وتُظهر** وضعَي فشل RAG (سؤال خارج المجموعة، k كبير جدًّا).
٣. **تختار k** لنظام RAG بناءً على جودة الاسترجاع وحجم المقطع.
٤. **تقارن** إجابة LLM عاري مقابل مُرسَّخة.

غدًا في **D3** الخطوة التالية: بدل أن نُعطي النموذج نصًّا ليقرأه، نُعطيه **أدوات
ليستعملها** — حاسبة، محرّك بحث، قارئ ملفّات — ونجعله يُخطّط لسلسلة أفعال. ذاك هو
الوكيل.

</div>
